In [1]:
!pip install pandas requests tqdm

In [2]:
import pandas as pd
import requests
from datetime import datetime, timedelta
from io import StringIO
from tqdm import tqdm

In [3]:
start_date = datetime(2025, 1, 1)
end_date   = datetime(2025, 12, 31)

In [4]:
BASE_URL = "https://www.spc.noaa.gov/climo/reports"

states_keep = {"IA", "IL", "MO"}

all_reports = []

In [5]:
current_date = start_date

while current_date <= end_date:
    datestr = current_date.strftime("%y%m%d")
    url = f"{BASE_URL}/{datestr}_rpts_torn.csv"

    try:
        response = requests.get(url, timeout=15)

        if response.status_code == 200:
            df = pd.read_csv(StringIO(response.text))

            # Filter states
            df = df[df["State"].isin(states_keep)]

            # Keep only desired columns (SPC column names)
            df = df[
                ["Time", "F_Scale", "Location", "County",
                 "State", "Lat", "Lon", "Comments"]
            ]

            # Add report date 
            df["Report_Date"] = current_date.strftime("%Y-%m-%d")

            all_reports.append(df)

        else:
            print(f"No file for {datestr}")

    except Exception as e:
        print(f"Error on {datestr}: {e}")

    current_date += timedelta(days=1)

In [6]:
if all_reports:
    tornado_2025 = pd.concat(all_reports, ignore_index=True)
else:
    tornado_2025 = pd.DataFrame()

tornado_2025.to_csv(
    "SPC_tornado_reports_2025_IA_IL_MO.csv",
    index=False
)